In [1]:
import xarray as xr
import pandas as pd
!pip install netCDF4 h5netcdf pydap
from pathlib import Path

urls_apr = "./modis_urls_0414-0418.txt"
urls_may_1 = "./modis_urls_0505-0517.txt"
urls_may_2 = "./modis_urls_0521-0528.txt"

with open(urls_apr, 'r') as f:
    data = f.read().splitlines()

sample_url = data[50]
sample_url = "https://ladsweb.modaps.eosdis.nasa.gov/opendap/RemoteResources/laads/allData/61/MYD04_L2/2010/107/MYD04_L2.A2010107.1135.061.2018059233002.hdf"


In [2]:
# helper class for Exceptions thrown for clarification

class InvalidValueError(Exception):
    pass

In [3]:
from glob import glob
# function that takes the date and outputs a concatenation of datasets from that date

# try_read = xr.open_dataset("data/raw/april_links/MOD04_L2.A2007104.1000.061.2017281024934.hdf", engine="netcdf4")
# try_read

def collect_dataset(date, day, type):
    # leap year check
    try:
        if (type == "Terra"):
            if (day >= 99 and day <= 114):
                files = sorted(glob(f"data/raw/april_links/MOD04*A{date}*.hdf"))
            elif (day >= 124 and day <= 139):
                files = sorted(glob(f"data/raw/may_links_1/MOD04*A{date}*.hdf"))
            elif (day >= 138 and day <= 153):
                files = sorted(glob(f"data/raw/may_links_2/MOD04*A{date}*.hdf"))
            else:
                print(f"Skipping {date}: Day {day} not in expected bounds.")
                return None
        elif (type == "Aqua"):
            if (day >= 99 and day <= 114):
                files = sorted(glob(f"data/raw/april_links/MYD04*A{date}*.hdf"))
            elif (day >= 124 and day <= 139):
                files = sorted(glob(f"data/raw/may_links_1/MYD04*A{date}*.hdf"))
            elif (day >= 138 and day <= 153):
                files = sorted(glob(f"data/raw/may_links_2/MYD04*A{date}*.hdf"))
            else:
                print(f"Skipping {date}: Day {day} not in expected bounds.")
                return None
        else:
            print("Error: The type must be either Terra or Aqua")
            return None
    except Exception as e:
        print(f"Error: {e}")
    datasets = []
    print(f"Processing {len(files)} files for {date}...")

    # VARIABLES KEPT
    # QA flags included in case important later
    keep_vars = [
        'Angstrom_Exponent_1_Ocean', 
        'Latitude', 
        'Longitude',
        'Quality_Assurance_Ocean',   
        'Land_Ocean_Quality_Flag'    
    ]

    for i, f in enumerate(files):
        try:
            ds = xr.open_dataset(f, engine="netcdf4")
            
            # filter variables, lots of dimension concatenation issues with height levels if not filtering
            actual_vars = [v for v in keep_vars if v in ds]
            ds = ds[actual_vars]

            # this allows sample count later in the script
            ds["swath_id"] = i

            # rename dimensions for easier recognition
            rename_dict = {
                'Cell_Along_Swath:mod04': 'track', 
                'Cell_Across_Swath:mod04': 'swath'
            }
            actual_rename = {k: v for k, v in rename_dict.items() if k in ds.dims}
            ds = ds.rename(actual_rename)

            datasets.append(ds)
        
        except Exception as e:
            print(f"Error loading {f}: {e}")

    if not datasets:
        print("No valid datasets found.")
        return None

    combined_swath = xr.concat(datasets, dim="track", coords='minimal')
    
    return combined_swath

In [4]:
import cartopy
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

def plot_angstrom(ds, title, save_path=False):
    all_lats = ds['Latitude'].values.ravel()
    all_lons = ds['Longitude'].values.ravel()
    all_ang = ds['Angstrom_Exponent_1_Ocean'].values.ravel()

    # Remove points where Angstrom Exponent is NaN (missing)
    mask = ~np.isnan(all_ang)
    all_lats = all_lats[mask]
    all_lons = all_lons[mask]
    all_ang = all_ang[mask]

    plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([-35, 10, 43, 71], crs=ccrs.PlateCarree())

    # Add map features
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

    # Add gridlines
    ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                linewidth=0.5, color='gray', alpha=0.5, linestyle='--')

    # Plot the points
    # Note: s=1 makes the dots tiny so they don't overlap too much
    sc = ax.scatter(all_lons, all_lats, c=all_ang, s=1, alpha=0.8, 
                    cmap='viridis', transform=ccrs.PlateCarree())

    # (Optional) Overlay the 1x1 Red Grid
    lat_bins = np.arange(43, 72, 1)
    lon_bins = np.arange(-35, 11, 1)

    for lat_edge in lat_bins:
        ax.plot([-35, 10], [lat_edge, lat_edge], color='red', linewidth=0.5, linestyle='--', transform=ccrs.PlateCarree())
    for lon_edge in lon_bins:
        ax.plot([lon_edge, lon_edge], [43, 71], color='red', linewidth=0.5, linestyle='--', transform=ccrs.PlateCarree())

    # Colorbar and Title
    cbar = plt.colorbar(sc, orientation='vertical', shrink=0.7)
    cbar.set_label('Angstrom Exponent (Ocean)')
    plt.title(f'Daily Composite: {title}', fontsize=16, fontweight="bold")

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig='all') # Closes the figure so it doesn't eat up all your RAM!
    else:
        plt.show()

In [5]:
def get_stats(ds, name):
    all_lats = ds['Latitude'].values.ravel()
    all_lons = ds['Longitude'].values.ravel()
    all_ang = ds['Angstrom_Exponent_1_Ocean'].values.ravel()
    ids_2d, _ = xr.broadcast(ds['swath_id'], ds['Angstrom_Exponent_1_Ocean'])
    all_ids = ids_2d.values.ravel()

    # Remove points where Angstrom Exponent is NaN (missing)
    mask = ~np.isnan(all_ang)
    all_lats = all_lats[mask]
    all_lons = all_lons[mask]
    all_ang = all_ang[mask]
    all_ids = all_ids[mask]

    lat_edges = np.arange(43, 72, 1)
    lon_edges = np.arange(-35, 11, 1)
    
    # Computing grid cell centers for plotting
    lat_centers = (lat_edges[:-1] + lat_edges[1:]) / 2
    lon_centers = (lon_edges[:-1] + lon_edges[1:]) / 2

    # Digitizing to assign points into the 1x1 grid boxes
    lat_idx = np.digitize(all_lats, lat_edges) - 1
    lon_idx = np.digitize(all_lons, lon_edges) - 1

    # Initializing arrays to accumulate sums and counts
    grid_sum_ang = np.zeros((len(lat_centers), len(lon_centers)))
    grid_count_ang = np.zeros_like(grid_sum_ang)

    # Accumulating sums and counts per cell
    for i in range(len(all_ang)):
        li = lat_idx[i]
        lj = lon_idx[i]
        if 0 <= li < len(lat_centers) and 0 <= lj < len(lon_centers):
            grid_sum_ang[li, lj] += all_ang[i]
            grid_count_ang[li, lj] += 1
    

    # Computing Angstrom Exponent daily mean
    grid_mean_ang = np.divide(grid_sum_ang, grid_count_ang, where=grid_count_ang>0)
    grid_mean_ang[grid_count_ang == 0] = np.nan  # empty cells are NaN

    # We create an empty grid to accumulate the counts
    swath_count_grid = np.zeros_like(grid_count_ang)
    
    # Loop through each unique swath ID (e.g., 0, 1, 2...)
    # This is fast because there are only ~15 swaths per day
    unique_swaths = np.unique(all_ids)
    
    for sid in unique_swaths:
        # Find pixels belonging to THIS swath
        swath_mask = (all_ids == sid)
        
        # Bin just this swath to see where it lands
        # We don't care about the values, just WHERE it is
        H, _, _ = np.histogram2d(all_lats[swath_mask], all_lons[swath_mask], bins=[lat_edges, lon_edges])
        
        # Wherever H > 0, this swath "touched" that grid box. Add 1.
        swath_count_grid += (H > 0).astype(int)

    # Creating xarray Dataset with mean and count
    daily_ds = xr.Dataset(
        {
            'Angstrom_mean': (('lat', 'lon'), grid_mean_ang),
            'sample_count': (('lat', 'lon'), grid_count_ang),
            'swath_count': (('lat', 'lon'), swath_count_grid)
        },
        coords={
            'lat': lat_centers,
            'lon': lon_centers
        }
    )

    return daily_ds

In [6]:
def plot_daily_mean(ds, title, save_path=False):
    lat = ds['lat'].values
    lon = ds['lon'].values
    angstrom_mean = ds['Angstrom_mean'].values  # 2D array

    # Defining the region of interest
    lat_min, lat_max = 43, 71
    lon_min, lon_max = -35, 10

    # Plotting daily mean Angstrom exponent in 1x1 grids
    plt.figure(figsize=(12,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())


    # Adding coastline, borders, and land features
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAND, facecolor='lightgray')

    # Adding lat/lon gridelines
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                    linewidth=1.2, color='black')

    # Creating 2D mesh for pcolormesh
    lon2d, lat2d = np.meshgrid(lon, lat)
    pcm = ax.pcolormesh(lon2d, lat2d, angstrom_mean, cmap='viridis', shading='auto', transform=ccrs.PlateCarree())

    lat_edges = np.arange(43, 72, 1)
    lon_edges = np.arange(-35, 11, 1)

    # Overlay 1x1 grid boxes in red dashed lines
    for lat_edge in lat_edges:
        ax.plot(lon_edges, [lat_edge]*len(lon_edges), color='red', linewidth=0.7, linestyle='--', transform=ccrs.PlateCarree())
    for lon_edge in lon_edges:
        ax.plot([lon_edge]*len(lat_edges), lat_edges, color='red', linewidth=0.7, linestyle='--', transform=ccrs.PlateCarree())

    # Colorbar
    cbar = plt.colorbar(pcm, orientation='vertical', shrink=0.7)
    cbar.set_label('Angstrom Exponent (Ocean)')

    # Title
    plt.title(f"MODIS Aqua Angstrom Exponent (Ocean): {title} 1°x1° Daily Mean", fontsize=16, fontweight="bold")
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig='all') # Closes the figure so it doesn't eat up all your RAM!
    else:
        plt.show()

In [7]:
def plot_sample_count(ds, title, save_path=False):
    # Same code but plotting sample counts instead of daily mean Angstrom Exponent
    # Extracting lat, lon, and sample counts
    lat = ds['lat'].values
    lon = ds['lon'].values
    sample_count = ds['sample_count'].values  # 2D array

    # Defining the region of interest
    lat_min, lat_max = 43, 71
    lon_min, lon_max = -35, 10

    # Defining 1x1 grid boxes
    lat_edges = np.arange(lat_min, lat_max + 1, 1)
    lon_edges = np.arange(lon_min, lon_max + 1, 1)

    # Plotting sample counts in 1x1 grids
    plt.figure(figsize=(12,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                    linewidth=1.2, color='black')

    lon2d, lat2d = np.meshgrid(lon, lat)
    pcm = ax.pcolormesh(lon2d, lat2d, sample_count, cmap='plasma', shading='auto', transform=ccrs.PlateCarree())

    for lat_edge in lat_edges:
        ax.plot(lon_edges, [lat_edge]*len(lon_edges), color='red', linewidth=0.7, linestyle='--', transform=ccrs.PlateCarree())
    for lon_edge in lon_edges:
        ax.plot([lon_edge]*len(lat_edges), lat_edges, color='red', linewidth=0.7, linestyle='--', transform=ccrs.PlateCarree())

    cbar = plt.colorbar(pcm, orientation='vertical', shrink=0.7)
    cbar.set_label('Number of Samples')

    plt.title(f"MODIS Aqua Sample Count: {title}", fontsize=16, fontweight="bold")
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig='all') # Closes the figure so it doesn't eat up all your RAM!
    else:
        plt.show()

In [8]:
def plot_swath_count(ds, title, save_path=False):
    # Same code but plotting sample counts instead of daily mean Angstrom Exponent
    # Extracting lat, lon, and sample counts
    lat = ds['lat'].values
    lon = ds['lon'].values
    swath_count = ds['swath_count'].values  # 2D array

    # Defining the region of interest
    lat_min, lat_max = 43, 71
    lon_min, lon_max = -35, 10

    # Defining 1x1 grid boxes
    lat_edges = np.arange(lat_min, lat_max + 1, 1)
    lon_edges = np.arange(lon_min, lon_max + 1, 1)

    # Plotting sample counts in 1x1 grids
    plt.figure(figsize=(12,8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                    linewidth=1.2, color='black')

    lon2d, lat2d = np.meshgrid(lon, lat)
    pcm = ax.pcolormesh(lon2d, lat2d, swath_count, cmap='plasma', shading='auto', transform=ccrs.PlateCarree())

    for lat_edge in lat_edges:
        ax.plot(lon_edges, [lat_edge]*len(lon_edges), color='red', linewidth=0.7, linestyle='--', transform=ccrs.PlateCarree())
    for lon_edge in lon_edges:
        ax.plot([lon_edge]*len(lat_edges), lat_edges, color='red', linewidth=0.7, linestyle='--', transform=ccrs.PlateCarree())

    cbar = plt.colorbar(pcm, orientation='vertical', shrink=0.7)
    cbar.set_label('Number of Swaths')

    plt.title(f"MODIS Aqua Swath Count {title}", fontsize=16, fontweight="bold")
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig='all') # Closes the figure so it doesn't eat up all your RAM!
    else:
        plt.show()

In [9]:
def create_climatology(dss, title, volcano, years):
    
    angstrom_cube = xr.concat(
        [ds["Angstrom_mean"] for ds in dss], dim = "year"
    )
    angstrom_cube['year'] = years
    
    valid_pixel_count = angstrom_cube.notnull().sum(dim="year")
    
    ### CHANGE FROM 6 TO DESIRED MINIMUM NUMBER OF DATA YEARS
    overlap_mask = (valid_pixel_count >= 4)

    clean_cube = angstrom_cube.where(overlap_mask)

    # 1. Base Stats
    clim_mean = clean_cube.mean(dim='year')
    clim_median = clean_cube.median(dim='year')
    clim_std = clean_cube.std(dim='year')

    # 2. Standard Error (SE = Standard Deviation / sqrt(N))
    # N is the number of valid years (which is 6, unless you relaxed the mask)
    clim_se = clim_std / np.sqrt(valid_pixel_count.where(overlap_mask))

    # 3. Confidence Intervals (Mean +/- SE)
    clim_mean_plus_1se = clim_mean + clim_se
    clim_mean_minus_1se = clim_mean - clim_se
    clim_mean_plus_2se = clim_mean + (2 * clim_se)
    clim_mean_minus_2se = clim_mean - (2 * clim_se)

    # 4. Bundle it all together into a final Climatology Dataset
    climatology_ds = xr.Dataset({
        'mean': clim_mean,
        'median': clim_median,
        'std_dev': clim_std,
        'std_error': clim_se,
        'mean_plus_1se': clim_mean_plus_1se,
        'mean_minus_1se': clim_mean_minus_1se,
        'mean_plus_2se': clim_mean_plus_2se,
        'mean_minus_2se': clim_mean_minus_2se,
        'valid_years_count': valid_pixel_count # Good to keep for reference
    })

    # This is your final baseline!
    print(climatology_ds)

    return clean_cube, climatology_ds


In [10]:
def plot_climatology(ds, volcano, title, years, save_path=False):

    # 1. Setup a 2-row, 3-column figure grid with Cartopy projections
    fig, axes = plt.subplots(
        nrows=2, ncols=3, 
        figsize=(18, 10), 
        subplot_kw={'projection': ccrs.PlateCarree()}
    )

    # Flatten the axes array to make it easy to loop through
    axes = axes.flatten()

    # Get coordinates for plotting
    lats = ds['lat'].values
    lons = ds['lon'].values

    # Define a fixed color scale so all 6 maps are comparable
    # (Adjust vmin and vmax based on your actual data ranges)
    vmin, vmax = 0.0, 2.0 

    # 2. Loop through the 6 years and plot each one
    for i, year in enumerate(years):
        ax = axes[i]
        
        # Extract the data for just this year
        # NOTE: We use 'clean_cube' so it only plots the overlapping pixels
        data = ds.sel(year=year).values
        
        # Add map features
        ax.set_extent([-35, 10, 43, 71], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        
        # Plot the heatmap
        pcm = ax.pcolormesh(
            lons, lats, data, 
            cmap='viridis', shading='auto', transform=ccrs.PlateCarree(),
            vmin=vmin, vmax=vmax # Lock color scale
        )
        
        ax.set_title(f"{title}, {year}", fontsize=14)

    # 3. Add a single, shared colorbar at the bottom
    # Create a new axis just for the colorbar to avoid shrinking the maps
    cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.03]) # [left, bottom, width, height]
    cbar = fig.colorbar(pcm, cax=cbar_ax, orientation='horizontal')
    cbar.set_label('Angstrom Exponent (Ocean)', fontsize=14)

    # 4. Add a main title and adjust spacing
    plt.suptitle(f'Climatology Baseline: {title}', fontsize=20, y=0.98)
    plt.subplots_adjust(bottom=0.15, hspace=0.2, wspace=0.1)

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig='all') # Closes the figure so it doesn't eat up all your RAM!
    else:
        plt.show()

In [11]:
import scipy.stats as stats

def compare_experimental(climate, ds):
    daily_mean = ds["Angstrom_mean"]

    df = climate["valid_years_count"] - 1

    t_crit_numpy = stats.t.ppf(0.975, df)
    t_crit = xr.DataArray(t_crit_numpy, coords=df.coords, dims=df.dims)

    upper_bound_t = climate["mean"] + (t_crit * climate["std_error"])
    lower_bound_t = climate["mean"] - (t_crit * climate["std_error"])

    upper_bound = climate["mean_plus_2se"]
    lower_bound = climate["mean_minus_2se"]

    is_fine_t = daily_mean > upper_bound_t
    is_coarse_t = daily_mean < lower_bound_t

    is_fine = daily_mean > upper_bound
    is_coarse = daily_mean < lower_bound

    fine_t = daily_mean.where(is_fine_t)
    coarse_t = daily_mean.where(is_coarse_t)

    fine = daily_mean.where(is_fine)
    coarse = daily_mean.where(is_coarse)

    return fine, coarse, fine_t, coarse_t



In [12]:
def plot_significant_deviations(fine_phase_mean, coarse_phase_mean, phase_title, save_path="./data/outputs_ttest/"):
    """
    Plots the Fine (Sulfate) and Coarse (Ash) anomalies side-by-side.
    """
    fig, axes = plt.subplots(
        nrows=1, ncols=2, 
        figsize=(16, 6), 
        subplot_kw={'projection': ccrs.PlateCarree()}
    )
    
    lats = fine_phase_mean['lat'].values
    lons = fine_phase_mean['lon'].values
    
    # --- Left Plot: Fine Mode (Sulfate) ---
    ax1 = axes[0]
    ax1.set_extent([-35, 10, 43, 71], crs=ccrs.PlateCarree())
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax1.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
    ax1.add_feature(cfeature.LAND, facecolor='lightgray')
    ax1.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    
    # Fine particles have HIGH Angstrom Exponent (often > 1.5)
    # Using 'Reds' to denote fine/sulfate
    pcm1 = ax1.pcolormesh(lons, lats, fine_phase_mean.values, cmap='Reds', shading='auto', transform=ccrs.PlateCarree(), vmin=1.0, vmax=2.5)
    ax1.set_title("Fine-mode Dominated (Sulfate-like)", fontsize=14)
    fig.colorbar(pcm1, ax=ax1, orientation='horizontal', pad=0.05, shrink=0.8, label='Angstrom Exponent')

    # --- Right Plot: Coarse Mode (Ash) ---
    ax2 = axes[1]
    ax2.set_extent([-35, 10, 43, 71], crs=ccrs.PlateCarree())
    ax2.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax2.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
    ax2.add_feature(cfeature.LAND, facecolor='lightgray')
    ax2.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    
    # Coarse particles have LOW Angstrom Exponent (often < 0.5)
    # Using 'Blues' to denote coarse/ash
    pcm2 = ax2.pcolormesh(lons, lats, coarse_phase_mean.values, cmap='Blues_r', shading='auto', transform=ccrs.PlateCarree(), vmin=0.0, vmax=1.0)
    ax2.set_title("Coarse-mode Dominated (Ash-like)", fontsize=14)
    fig.colorbar(pcm2, ax=ax2, orientation='horizontal', pad=0.05, shrink=0.8, label='Angstrom Exponent')

    plt.suptitle(f'Statistically Significant Aerosol Deviations: {phase_title}', fontsize=18, fontweight='bold', y=1.02)
    if save_path:
        plt.savefig(f"{save_path}{phase_title}", bbox_inches='tight', dpi=300)
        plt.close(fig='all') # Closes the figure so it doesn't eat up all your RAM!
    else:
        plt.show()

In [13]:
# Figure out why not continuous plot

def plot_daily_anomaly(exp_stat, climate, title, save_path=False):
    """
    Plots the continuous anomaly (Experimental Daily Mean - Climatological Phase Mean).
    Uses a diverging colormap centered at 0.
    """
    # 1. Calculate the raw difference
    anomaly = exp_stat['Angstrom_mean'] - climate['mean']
    
    lats = exp_stat['lat'].values
    lons = exp_stat['lon'].values
    
    plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([-35, 10, 43, 71], crs=ccrs.PlateCarree())
    
    # Map features
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                 linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
    
    # 2. Plot using a diverging colormap ('RdBu_r' or 'coolwarm')
    # vmin and vmax MUST be symmetrical (e.g., -1.5 to 1.5) so white is exactly 0
    pcm = ax.pcolormesh(
        lons, lats, anomaly.values, 
        cmap='RdBu_r', shading='auto', transform=ccrs.PlateCarree(),
        vmin=-1.5, vmax=1.5 
    )
    
    # Optional gridlines to match your other plots
    lat_edges = np.arange(43, 72, 1)
    lon_edges = np.arange(-35, 11, 1)
    for lat_edge in lat_edges:
        ax.plot(lon_edges, [lat_edge]*len(lon_edges), color='black', linewidth=0.3, linestyle='--', transform=ccrs.PlateCarree(), alpha=0.3)
    for lon_edge in lon_edges:
        ax.plot([lon_edge]*len(lat_edges), lat_edges, color='black', linewidth=0.3, linestyle='--', transform=ccrs.PlateCarree(), alpha=0.3)

    cbar = plt.colorbar(pcm, orientation='vertical', shrink=0.7)
    cbar.set_label('Angstrom Exponent Anomaly (Daily - Baseline)')
    
    plt.title(title, fontsize=16, fontweight='bold')
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        plt.close(fig='all') # Closes the figure so it doesn't eat up all your RAM!
    else:
        plt.show()

In [14]:
# from datetime import datetime as dt

# def run_pipeline(day, satellite):

#     title = f"{satellite}: {day[0:4]} Day {day[4:7]}"

#     n_day = int(day[4:7])
#     print(n_day)

#     ds = collect_dataset(day, n_day, satellite)
#     plot_angstrom(ds, f"{title} Angstrom Exponent")
#     stat_ds = get_stats(ds, "")
#     print(stat_ds)
#     plot_daily_mean(stat_ds, title)
#     plot_sample_count(stat_ds, title)
#     plot_swath_count(stat_ds, title)

# def run_climatology(day, satellite):
#     dss = []
#     stat_dss = []
#     n_day = int(day[4:7])
#     clean_day = f"Day {n_day}"
#     years = [2007, 2008, 2009, 2012, 2013, 2014]
#     if ((n_day >= 101 and n_day <= 110) or (n_day >= 122 and n_day <= 139 and int(day[0:4]) == 2010)):
#         volcano = "Eja"
#     else: 
#         volcano = "Grims"
#     for year in years:
#         dss.append(collect_dataset(f"{year}{day[4:7]}", n_day, satellite))
    
#     for ds in dss:
#         stat_dss.append(get_stats(ds, ""))

#     clean_cube, climate = create_climatology(stat_dss, volcano, f"{volcano} Climatology", years)
#     plot_climatology(clean_cube, volcano, f"{volcano} Climatology {clean_day} ({satellite})", years)

#     print(climate)

#     return climate

# def run_experimental(day, satellite):
#     climate = run_climatology(day, satellite)

#     n_day = int(day[4:7])

#     if ((n_day >= 101 and n_day <= 110) or (n_day >= 122 and n_day <= 139 and int(day[0:4]) == 2010)):
#         volcano = "Eja"
#     else: 
#         volcano = "Grims"
    
#     exp_ds = collect_dataset(day, n_day, satellite)
#     plot_angstrom(exp_ds, f"{satellite}: Day {n_day} Angstrom Exponent")
#     exp_stat = get_stats(exp_ds, "")

#     fine, coarse = compare_experimental(climate, exp_stat)
#     plot_significant_deviations(fine, coarse, f"{satellite}: {volcano} Day {n_day} AE Anomalies")

from datetime import datetime as dt, timedelta

def run_pipeline(year, month, day_str, satellite):
    # Calculate Julian day for this specific year
    date_obj = dt.strptime(f"{month}/{day_str}/{year}", "%m/%d/%Y")
    j_day = date_obj.strftime("%j")
    n_day = int(j_day)

    title = f"{satellite}: {year} Day {j_day}"
    print(f"Processing Day of Year: {n_day}")

    ds = collect_dataset(f"{year}{j_day}", n_day, satellite)
    if ds is not None:
        plot_angstrom(ds, f"{title} Angstrom Exponent")
        stat_ds = get_stats(ds, "")
        plot_daily_mean(stat_ds, title)
        plot_sample_count(stat_ds, title)
        plot_swath_count(stat_ds, title)

def get_phase_dates(year, start_md, end_md):
    """
    Takes a year and start/end dates (e.g., '04/14', '04/18') 
    and returns a list of datetime objects for that range.
    """
    start = dt.strptime(f"{start_md}/{year}", "%m/%d/%Y")
    end = dt.strptime(f"{end_md}/{year}", "%m/%d/%Y")
    delta = end - start
    return [start + timedelta(days=i) for i in range(delta.days + 1)]

def get_yearly_phase_average(year, start_md, end_md, satellite):
    """
    Averages the days within a phase for a single year.
    Enforces the constraint: >= 4 valid days per grid box.
    """
    dates = get_phase_dates(year, start_md, end_md)
    daily_stats = []
    
    for d in dates:
        j_day = d.strftime("%j")
        n_day = int(j_day)
        
        # Collect the raw data
        ds = collect_dataset(f"{year}{j_day}", n_day, satellite)
        if ds is not None:
            # Grid the data
            stat = get_stats(ds, "")
            daily_stats.append(stat["Angstrom_mean"])
            
    if not daily_stats:
        return None
        
    # Stack the days for this year into a cube
    phase_cube = xr.concat(daily_stats, dim='day')
    
    # CONSTRAINT: Grid point must have at least 4 valid days
    valid_days_count = phase_cube.notnull().sum(dim='day')
    mask_4days = valid_days_count >= 4
    
    # Calculate the mean and apply the mask
    phase_mean = phase_cube.mean(dim='day', skipna=True).where(mask_4days)
    
    # Wrap it back into a Dataset so it matches what create_climatology expects
    return xr.Dataset({'Angstrom_mean': phase_mean, 'lat': stat['lat'], 'lon': stat['lon']})


# def run_climatology(year, month, day_str, satellite):
#     dss = []
#     stat_dss = []
#     years = [2007, 2008, 2009, 2012, 2013, 2014]
    
#     # We use the month/day to determine the volcano rather than Julian day
#     if month == "04":
#         volcano = "Eja"
        
#     elif (month == "05" and int(day_str) <= 19):
#         volcano = "Eja"
#     else: 
#         volcano = "Grims"
        
#     clean_day = f"{month}/{day_str}"

#     for year in years:
#         #calculate Julian day dynamically for each climatology year
#         date_obj = dt.strptime(f"{month}/{day_str}/{year}", "%m/%d/%Y")
#         j_day = date_obj.strftime("%j")
#         n_day = int(j_day)
        
#         dss.append(collect_dataset(f"{year}{j_day}", n_day, satellite))
    
#     for ds in dss:
#         if ds is not None: # safeguard
#             stat_dss.append(get_stats(ds, ""))

#     clean_cube, climate = create_climatology(stat_dss, volcano, f"{volcano} Climatology", years)
#     plot_climatology(clean_cube, volcano, f"{volcano} Climatology {clean_day} ({satellite})", years)

#     return climate

def run_phase_climatology(start_md, end_md, satellite, plot_yearly=True):
    yearly_phase_dss = []
    valid_years = []
    years = [2007, 2008, 2009, 2012, 2013, 2014]
    
    # Determine volcano for titles
    month = start_md.split('/')[0]
    day = int(start_md.split('/')[1])
    if month == "04" or (month == "05" and day <= 19):
        volcano = "Eja"
    else: 
        volcano = "Grims"

    print(f"\n--- Building Phase Climatology for {volcano} ({start_md} to {end_md}) ---")
    
    for year in years:
        print(f"Aggregating Phase for Year {year}...")
        phase_ds = get_yearly_phase_average(year, start_md, end_md, satellite)
        
        if phase_ds is not None:
            yearly_phase_dss.append(phase_ds)
            valid_years.append(year)

    # Use your exact existing function to compute the 6-year baseline!
    clean_cube, climate = create_climatology(yearly_phase_dss, volcano, f"{volcano} Phase Climatology", valid_years)
    
    # Plot the 6-panel grid (now showing the Phase average for each year)
    if plot_yearly:
        plot_climatology(clean_cube, volcano, f"{volcano} Phase {start_md}-{end_md} ({satellite})", valid_years)

    return climate


# def run_experimental(year, month, day_str, satellite):
#     # Run baseline (it will dynamically handle the leap years inside)
#     climate = run_climatology(month, day_str, satellite)

#     # Get the Julian day for the specific experimental year
#     date_obj = dt.strptime(f"{month}/{day_str}/{year}", "%m/%d/%Y")
#     j_day = date_obj.strftime("%j")
#     n_day = int(j_day)

#     if month == "04" or (month == "05" and int(day_str) <= 19):
#         volcano = "Eja"
#     else: 
#         volcano = "Grims"
    
#     exp_ds = collect_dataset(f"{year}{j_day}", n_day, satellite)
#     if exp_ds is not None:
#         plot_angstrom(exp_ds, f"{satellite}: Day {n_day} Angstrom Exponent")
#         exp_stat = get_stats(exp_ds, "")

#         fine, coarse = compare_experimental(climate, exp_stat)
#         plot_significant_deviations(fine, coarse, f"{satellite}: {volcano} Day {n_day} AE Anomalies")

def run_experimental(year, month, day_str, phase_start, phase_end, satellite):
    # 1. Generate the Phase-averaged Climatology
    climate = run_phase_climatology(phase_start, phase_end, satellite)

    # 2. Process the specific experimental DAY
    date_obj = dt.strptime(f"{month}/{day_str}/{year}", "%m/%d/%Y")
    j_day = date_obj.strftime("%j")
    n_day = int(j_day)

    if month == "04" or (month == "05" and int(day_str) <= 19):
        volcano = "Eja"
    else: 
        volcano = "Grims"
    
    exp_ds = collect_dataset(f"{year}{j_day}", n_day, satellite)
    if exp_ds is not None:
        plot_angstrom(exp_ds, f"{satellite}: Day {n_day} Angstrom Exponent")
        exp_stat = get_stats(exp_ds, "")

        plot_daily_anomaly(exp_stat, climate, f"{satellite}: {volcano} Day {n_day} Anomaly Values")

        # 3. Compare the experimental DAY to the PHASE Baseline
        fine, coarse, fine_t, coarse_t = compare_experimental(climate, exp_stat)
        plot_significant_deviations(fine, coarse, f"{satellite}: {volcano} Day {n_day} Actual Significant Values")
    

# user_input = input("Type the day of interest here: (format yyyymmdd): ")
# try:
#     year = user_input[0:4]
#     month = user_input[4:6]
#     day = user_input[6:8]
    
#     # Just to validate it's a real date
#     dt.strptime(f"{month}/{day}/{year}", "%m/%d/%Y")
# except Exception as e:
#     print(f"Error reading date: {e}")

# action_input = input("Which type of analysis are you running (options: daily plots (d), climatology (c), full analysis (f))? ")
# tod = input("Which satellite are you interested in (Aqua (a), Terra (t))? ")

# try:
#     if tod == "a":
#         sat_name = "Aqua"
#     elif tod == "t":
#         sat_name = "Terra"
#     else:
#         raise Exception("You must specify either Aqua (type a) or Terra (type t)")

#     if action_input == "d":
#         run_pipeline(year, month, day, sat_name)
#     elif action_input == "c":
#         # Hardcode the phase boundaries based on the month/day input
#         if month == "04":
#             phase_start, phase_end = "04/09", "04/23"
#         elif month == "05" and int(day) <= 19:
#             phase_start, phase_end = "05/04", "05/18"
#         elif month == "05" and int(day) > 19:
#             phase_start, phase_end = "05/18", "06/01"
#         else:
#             print("Invalid date for climatology.")
            
#         run_phase_climatology(phase_start, phase_end, sat_name)
#         # run_climatology(year, month, day, sat_name)
#     elif action_input == "f":
#         if year in ["2010", "2011"]:
#             # Hardcode the phase boundaries based on the month/day input
#             if month == "04":
#                 phase_start, phase_end = "04/09", "04/23"
#             elif month == "05" and int(day) <= 19:
#                 phase_start, phase_end = "05/04", "05/18"
#             elif month == "05" and int(day) > 19:
#                 phase_start, phase_end = "05/18", "06/01"
#             else:
#                 print("You can't use that date for the full experimental pipeline \n")
                
#             run_experimental(year, month, day, phase_start, phase_end, sat_name)
#         else:
#             raise Exception("You chose full analysis... try entering an experimental year (2010, 2011)")
#     else:
#         raise Exception("Try typing one of the allowed letters for analysis type (d, c, f).")

# except Exception as e:
#     print(f"Error: {e}")
    
    
    

In [15]:
from IPython.display import clear_output

def plot_triple_climatology(satellite):
    print(f"\n========== Generating 3-Phase Master Climatology for {satellite} ==========")
    
    # 1. Generate the baseline stats for all three phases
    clim_eyja1 = run_phase_climatology("04/09", "04/23", satellite, plot_yearly=False)
    clim_eyja2 = run_phase_climatology("05/04", "05/18", satellite, plot_yearly=False)
    clim_grims = run_phase_climatology("05/18", "06/01", satellite, plot_yearly=False)
    
    # 2. Extract the overall 'mean' variable from each phase
    # (These represent the 6-year aggregated climatology for the padded phases)
    clear_output(wait=True)
    means = [
        clim_eyja1['mean'], 
        clim_eyja2['mean'], 
        clim_grims['mean']
    ]
    
    titles = [
        "Eyjafjallajökull Phase 1\n(Apr 09 - Apr 23)",
        "Eyjafjallajökull Phase 2\n(May 04 - May 18)",
        "Grímsvötn\n(May 18 - Jun 01)"
    ]

    # 3. Setup the 1x3 plot grid
    fig, axes = plt.subplots(
        nrows=1, ncols=3, 
        figsize=(20, 7), 
        subplot_kw={'projection': ccrs.PlateCarree()}
    )
    
    # Lock the color scale so the three phases are comparable visually
    vmin, vmax = 0.0, 2.0 
    
    for i in range(3):
        ax = axes[i]
        data = means[i].values
        lats = means[i]['lat'].values
        lons = means[i]['lon'].values
        
        # Add map features
        ax.set_extent([-35, 10, 43, 71], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5)
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.gridlines(draw_labels=False, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        
        # Plot the heatmap
        pcm = ax.pcolormesh(
            lons, lats, data, 
            cmap='viridis', shading='auto', transform=ccrs.PlateCarree(),
            vmin=vmin, vmax=vmax 
        )
        
        ax.set_title(titles[i], fontsize=15, fontweight='bold', pad=10)

    # 4. Add a single, shared colorbar spanning the bottom
    cbar_ax = fig.add_axes([0.15, 0.08, 0.7, 0.04]) # [left, bottom, width, height]
    cbar = fig.colorbar(pcm, cax=cbar_ax, orientation='horizontal')
    cbar.set_label('Climatological Mean Angstrom Exponent', fontsize=14)

    plt.suptitle(f'Master Climatology Baselines ({satellite})', fontsize=22, y=1.05)
    plt.subplots_adjust(bottom=0.2, wspace=0.1)
    
    plt.show()
    print("========== Finished! ==========\n")

In [16]:
import os

def run_and_save_all_experiments(satellite, base_dir="./data/outputs_ttest"):
    # Define the boundaries for the three experiments
    phases = [
        {"year": "2010", "volcano": "Eja", "erupt_start": "04/14", "erupt_end": "04/18", "clim_start": "04/09", "clim_end": "04/23"},
        {"year": "2010", "volcano": "Eja", "erupt_start": "05/05", "erupt_end": "05/17", "clim_start": "05/04", "clim_end": "05/18"},
        {"year": "2011", "volcano": "Grims", "erupt_start": "05/21", "erupt_end": "05/28", "clim_start": "05/18", "clim_end": "06/01"}
    ]

    print(f"\n=======================================================")
    print(f" STARTING BULK AUTOMATION FOR {satellite.upper()}")
    print(f"=======================================================\n")

    for phase in phases:
        print(f"\n>>> Generating Baseline: {phase['volcano']} Phase {phase['erupt_start']} to {phase['erupt_end']}")
        
        # 1. Generate the baseline for this phase ONCE
        climate = run_phase_climatology(phase['clim_start'], phase['clim_end'], satellite, plot_yearly=False)

        # 2. Get the specific daily dates for the eruption
        dates = get_phase_dates(phase['year'], phase['erupt_start'], phase['erupt_end'])

        for d in dates:
            date_str = d.strftime("%Y%m%d") # e.g., '20100414'
            j_day = d.strftime("%j")
            n_day = int(j_day)
            
            clear_output(wait=True)
            print(f"\n--- Processing Date: {date_str} (Day {n_day}) ---")

            # 3. Create the unique directory for this specific date
            day_dir = os.path.join(base_dir, satellite, date_str)
            os.makedirs(day_dir, exist_ok=True)

            # 4. Collect the data
            exp_ds = collect_dataset(f"{phase['year']}{j_day}", n_day, satellite)
            if exp_ds is None:
                print(f"Skipping {date_str} - No data found.")
                continue
                
            exp_stat = get_stats(exp_ds, "")

            # 5. Generate and SAVE the plots
            # (Make sure you updated these functions with the save_path block!)
            plot_angstrom(exp_ds, f"{satellite}: {date_str} Raw AE", save_path=os.path.join(day_dir, "01_raw_scatter.png"))
            plot_daily_mean(exp_stat, f"{satellite}: {date_str}", save_path=os.path.join(day_dir, "02_daily_mean.png"))
            plot_sample_count(exp_stat, f"{satellite}: {date_str}", save_path=os.path.join(day_dir, "03_sample_count.png"))
            plot_swath_count(exp_stat, f"{satellite}: {date_str}", save_path=os.path.join(day_dir, "04_swath_count.png"))
            
            # 6. Generate Anomaly and T-Test Thresholds
            plot_daily_anomaly(exp_stat, climate, f"{satellite}: {date_str} Anomaly", save_path=os.path.join(day_dir, "05_continuous_anomaly.png"))
            
            fine, coarse, fine_t, coarse_t = compare_experimental(climate, exp_stat)
            plot_significant_deviations(fine_t, coarse_t, f"{satellite}: {date_str} Significant Deviations", save_path=os.path.join(day_dir, "06_ttest_significance.png"))

            # 7. SAVE THE ACTUAL T-TEST DATA
            # We save the resulting DataArrays to NetCDF files so you have the raw mathematical output
            fine_t.to_netcdf(os.path.join(day_dir, "data_significant_fine.nc"))
            coarse_t.to_netcdf(os.path.join(day_dir, "data_significant_coarse.nc"))
            exp_stat.to_netcdf(os.path.join(day_dir, "data_daily_stats.nc"))
            
            print(f"Successfully saved all plots and data to: {day_dir}")

    print("\n=======================================================")
    print(" ALL EXPERIMENTS COMPLETE")
    print("=======================================================\n")

In [17]:
# run_and_save_all_experiments("Aqua")
# run_and_save_all_experiments("Terra")

In [18]:
# plot_triple_climatology("Aqua")
# plot_triple_climatology("Terra")

In [19]:
def run_phase_aggregation(year, erupt_start, erupt_end, clim_start, clim_end, satellite):
    fine_daily_grids = []
    coarse_daily_grids = []
    
    if (year == "2010"):
        if (erupt_start == "04/14"):
            volcano = "Eyjafjallajökull_p1"
        else:
            volcano = "Eyjafjallajökull_p2"
    else: 
        volcano = "Grímsvötn"

    phase_title = f"{satellite}: {volcano} Plume Core (≥3 Days Consecutive)"
    print(f"\n========== Starting {phase_title} ==========")

    # 1. Get the single Phase Climatology Baseline ONCE
    climate = run_phase_climatology(clim_start, clim_end, satellite, plot_yearly=False)

    # 2. Get the specific dates for the eruption
    eruption_dates = get_phase_dates(year, erupt_start, erupt_end)

    for d in eruption_dates:
        j_day = d.strftime("%j")
        n_day = int(j_day)
        
        print(f"\n--- Processing {d.strftime('%m/%d/%Y')} (Day {n_day}) ---")
        
        # Collect Experimental Data
        exp_ds = collect_dataset(f"{year}{j_day}", n_day, satellite)
        if exp_ds is None:
            print(f"Skipping {d.strftime('%m/%d')} - No data found.")
            continue
            
        exp_stat = get_stats(exp_ds, "")
        
        # 3. Compare daily data against the PHASE baseline
        fine, coarse, fine_t, coarse_t = compare_experimental(climate, exp_stat)
        
        fine_daily_grids.append(fine_t)
        coarse_daily_grids.append(coarse_t)
        
    # --- AGGREGATION & FILTERING STEP ---
    if not fine_daily_grids:
        print("No valid data found for the entire phase.")
        return

    print(f"\nAggregating and isolating the plume core...")
    
    # Stack the daily grids into 3D cubes (day, lat, lon)
    fine_cube = xr.concat(fine_daily_grids, dim='day')
    coarse_cube = xr.concat(coarse_daily_grids, dim='day')
    
    # Calculate the raw phase means (for the fine particles)
    fine_phase_mean = fine_cube.mean(dim='day', skipna=True)
    coarse_phase_mean = coarse_cube.mean(dim='day', skipna=True)
    
    # --- METHOD 2: THE PERSISTENCE & SPATIAL FILTERS (COARSE ONLY) ---
    
    # Filter 1: 3-Day Consecutive Rolling Window
    # Convert to 1s and 0s
    binary_coarse = coarse_cube.notnull().fillna(0).astype(int)
    # Sum rolling 3-day windows
    rolling_3day_sum = binary_coarse.rolling(day=3, center=False).sum()
    # Collapse to a 2D map: True if ANY 3-day window hit 3 consecutive days
    if (erupt_start == "04/14"):
        phase = "p1"
        is_continuous_3day = (rolling_3day_sum == 2).any(dim='day')
    else:
        phase = "p2"
        is_continuous_3day = (rolling_3day_sum == 2).any(dim='day')
    
    # Filter 2: Spatial Contiguity (Island Remover)
    # Roll a 3x3 box over the boolean map. Must have at least 4 contiguous pixels (itself + 3 neighbors)
    spatial_neighbors = is_continuous_3day.astype(int).rolling(lat=3, lon=3, center=True).sum()
    is_massive_plume = is_continuous_3day & (spatial_neighbors >= 4)
    
    # Apply the final strict mask to the coarse mean
    final_plume_core = coarse_phase_mean.where(is_massive_plume)
    
    # Plot the results
    # Left plot: Standard fine anomalies
    # Right plot: The strictly filtered coarse plume core
    plot_significant_deviations(fine_phase_mean, final_plume_core, phase_title)
    
    # Save the final mask so you know exactly which latitudes/longitudes to target for cloud data!
    final_plume_core.to_netcdf(f"./data/outputs_ttest/{satellite}_{volcano}_{phase}_coarse_bounding_box.nc")
    
    print(f"========== Finished {phase_title} ==========\n")

In [20]:
# 1. Eyjafjallajökull Phase 1
run_phase_aggregation("2010", "04/14", "04/18", "04/09", "04/23", "Aqua")
run_phase_aggregation("2010", "04/14", "04/18", "04/09", "04/23", "Terra")

# 2. Eyjafjallajökull Phase 2
run_phase_aggregation("2010", "05/05", "05/17", "05/04", "05/18", "Aqua")
run_phase_aggregation("2010", "05/05", "05/17", "05/04", "05/18", "Terra")

# 3. Grímsvötn
run_phase_aggregation("2011", "05/21", "05/28", "05/18", "06/01", "Aqua")
run_phase_aggregation("2011", "05/21", "05/28", "05/18", "06/01", "Terra")


========== Starting Aqua: Eyjafjallajökull_p1 Plume Core (≥3 Days Consecutive) ==========

--- Building Phase Climatology for Eja (04/09 to 04/23) ---
Aggregating Phase for Year 2007...
Processing 17 files for 2007099...


/tmp/ipykernel_1766787/579529155.py:76: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  combined_swath = xr.concat(datasets, dim="track", coords='minimal')
/tmp/ipykernel_1766787/347466235.py:40: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  grid_mean_ang = np.divide(grid_sum_ang, grid_count_ang, where=grid_count_ang>0)


Processing 16 files for 2007100...


/tmp/ipykernel_1766787/579529155.py:76: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  combined_swath = xr.concat(datasets, dim="track", coords='minimal')
/tmp/ipykernel_1766787/347466235.py:40: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  grid_mean_ang = np.divide(grid_sum_ang, grid_count_ang, where=grid_count_ang>0)


Processing 19 files for 2007101...


...


--- Processing 05/27/2011 (Day 147) ---
Processing 19 files for 2011147...


/tmp/ipykernel_1766787/579529155.py:76: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  combined_swath = xr.concat(datasets, dim="track", coords='minimal')
/tmp/ipykernel_1766787/347466235.py:40: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  grid_mean_ang = np.divide(grid_sum_ang, grid_count_ang, where=grid_count_ang>0)



--- Processing 05/28/2011 (Day 148) ---
Processing 19 files for 2011148...


/tmp/ipykernel_1766787/579529155.py:76: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  combined_swath = xr.concat(datasets, dim="track", coords='minimal')
/tmp/ipykernel_1766787/347466235.py:40: UserWarning: 'where' used without 'out', expect unitialized memory in output. If this is intentional, use out=None.
  grid_mean_ang = np.divide(grid_sum_ang, grid_count_ang, where=grid_count_ang>0)



Aggregating and isolating the plume core...
========== Finished Terra: Grímsvötn Plume Core (≥3 Days Consecutive) ==========

